# SQL-Based Data Analysis

This phase focuses on using SQL queries to analyze logistics data.  
The cleaned datasets from Phase 2 are loaded into a SQL database and queried to generate structured insights.

Key objectives of this phase:

• Establish SQL query foundations  
• Apply filtering, sorting, and grouping  
• Perform joins across datasets  
• Create summary tables for analysis  
• Extract insights from query results

## 1. Database Connection

In this step, a SQLite database is created to store the cleaned logistics datasets.

SQLite is used because it is lightweight and allows us to execute SQL queries directly from Python.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path


from pathlib import Path

BASE = Path.cwd()

# Support running from the project root or notebooks/ folder
if not (BASE / "data").is_dir():
    if (BASE.parent / "data").is_dir():
        BASE = BASE.parent
    else:
        raise FileNotFoundError(
            "Project data folder not found. "
            "Run this notebook from the project root or notebooks folder."
        )

RAW = BASE / "data" / "raw"
PROC = BASE / "data" / "processed"
DB = BASE / "data" / "logistics.db"

# Create only the directory needed for processed datasets
PROC.mkdir(parents=True, exist_ok=True)

conn = sqlite3.connect(DB)

print("Project root:", BASE)
print("Raw data:", RAW)
print("Processed data:", PROC)

Project root: d:\Athar\LogiTech_Logistics_Analysis
Raw data: d:\Athar\LogiTech_Logistics_Analysis\data\raw
Processed data: d:\Athar\LogiTech_Logistics_Analysis\data\processed


## 2. Loading Cleaned Data into SQL Tables

The cleaned datasets generated in Phase 2 are loaded into SQL tables.

Two tables are created:

• shipments    
• warehouses

These tables will be used for SQL queries and analysis.

In [2]:
shipment = pd.read_csv(PROC / "shipment_cleaned.csv")
warehouse = pd.read_csv(PROC / "warehouse_cleaned.csv")

shipment.to_sql("shipments", conn, if_exists="replace", index=False)
warehouse.to_sql("warehouses", conn, if_exists="replace", index=False)

print("SQL tables created successfully.")

SQL tables created successfully.


## 3. SQL Query Foundations

This query demonstrates basic SQL operations including:

• filtering records  
• sorting results  
• selecting relevant columns

The query retrieves shipments with high shipment costs.

In [3]:
query = """
SELECT
    Shipment_ID,
    Warehouse_ID,
    Weight,
    Shipment_Cost
FROM shipments
WHERE Shipment_Cost > 100
ORDER BY Shipment_Cost DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,Shipment_ID,Warehouse_ID,Weight,Shipment_Cost
0,S000885,PLANT16,2338.405126,31050.278620
1,S000886,PLANT16,749.400000,9201.432960
2,S000884,PLANT16,527.534472,7004.813739
3,S000887,PLANT16,360.800000,4790.846720
4,S000859,PLANT16,133.224269,1635.780861
5,S000888,PLANT16,68.035276,835.364332
6,S000861,PLANT16,58.774223,721.653420
7,S000858,PLANT16,42.830667,568.722732
8,S000866,PLANT16,37.987381,466.424255
9,S000857,PLANT16,37.041956,454.815954


## 4. Aggregation Queries

Aggregation functions allow us to summarize operational data.

This query calculates:

• average delivery time per warehouse  
• total number of shipments handled by each warehouse

In [4]:
query = """
SELECT
    Warehouse_ID,
    AVG(Delivery_Time_Days) AS Avg_Delivery_Time,
    COUNT(*) AS Total_Shipments
FROM shipments
GROUP BY Warehouse_ID
ORDER BY Avg_Delivery_Time DESC
LIMIT 10
"""

pd.read_sql(query, conn)

,Warehouse_ID,Avg_Delivery_Time,Total_Shipments
0,PLANT13,3.000000,86
1,PLANT09,3.000000,12
2,PLANT08,3.000000,102
3,PLANT03,2.807517,8541
4,PLANT16,2.693642,173
5,PLANT12,2.623333,300
6,PLANT04,2.000000,1


## 5. Joins and Data Integration

Joins allow us to combine multiple datasets.

In this query:

• shipment data is combined with warehouse operational metrics  
• delivery performance is compared with warehouse utilization

In [5]:
query = """
SELECT
    s.Warehouse_ID,
    AVG(w."Warehouse_Utilization_%") AS Avg_Warehouse_Utilization,
    AVG(s.Delivery_Time_Days) AS Avg_Delivery_Time
FROM shipments s
INNER JOIN warehouses w
    ON s.Warehouse_ID = w.Warehouse_ID
GROUP BY s.Warehouse_ID
ORDER BY Avg_Delivery_Time DESC
"""

pd.read_sql(query, conn)

,Warehouse_ID,Avg_Warehouse_Utilization,Avg_Delivery_Time
0,PLANT13,55.02,3.000000
1,PLANT09,61.69,3.000000
2,PLANT08,54.04,3.000000
3,PLANT03,71.68,2.807517
4,PLANT16,51.97,2.693642
5,PLANT12,67.85,2.623333
6,PLANT04,67.35,2.000000


## 6. Summary Table Creation

Summary tables help organize data for repeated analysis.

A SQL view is created to summarize shipment performance by warehouse.

In [6]:
conn.execute("DROP VIEW IF EXISTS warehouse_summary")
print("Old view removed successfully.")

Old view removed successfully.


In [7]:
query = """
CREATE VIEW warehouse_summary AS
SELECT
    s.Warehouse_ID,
    COUNT(*) AS Total_Shipments,
    AVG(s.Delivery_Time_Days) AS Avg_Delivery_Time,
    AVG(s.Shipment_Cost) AS Avg_Cost,
    AVG(w."Warehouse_Utilization_%") AS Avg_Warehouse_Utilization
FROM shipments s
LEFT JOIN warehouses w
    ON s.Warehouse_ID = w.Warehouse_ID
GROUP BY s.Warehouse_ID
"""

conn.execute(query)
print("View warehouse_summary created successfully.")

View warehouse_summary created successfully.


## 7. Retrieving Summary Table

The previously created SQL view is queried to analyze warehouse performance.

In [8]:
pd.read_sql("SELECT * FROM warehouse_summary", conn)

,Warehouse_ID,Total_Shipments,Avg_Delivery_Time,Avg_Cost,Avg_Warehouse_Utilization
0,PLANT03,8541,2.807517,7.570825,71.68
1,PLANT04,1,2.000000,NaN,67.35
2,PLANT08,102,3.000000,1.849302,54.04
3,PLANT09,12,3.000000,3.331973,61.69
4,PLANT12,300,2.623333,9.492059,67.85
5,PLANT13,86,3.000000,3.516375,55.02
6,PLANT16,173,2.693642,1743.995452,51.97


## 8. Insight Extraction

SQL queries help identify operational patterns in logistics data.

This query identifies warehouses with higher shipment costs and delivery times.

In [9]:
pd.read_sql("""
SELECT *
FROM warehouse_summary
ORDER BY Avg_Cost DESC
""", conn)

,Warehouse_ID,Total_Shipments,Avg_Delivery_Time,Avg_Cost,Avg_Warehouse_Utilization
0,PLANT16,173,2.693642,1743.995452,51.97
1,PLANT12,300,2.623333,9.492059,67.85
2,PLANT03,8541,2.807517,7.570825,71.68
3,PLANT13,86,3.000000,3.516375,55.02
4,PLANT09,12,3.000000,3.331973,61.69
5,PLANT08,102,3.000000,1.849302,54.04
6,PLANT04,1,2.000000,NaN,67.35


## Key Insights from SQL Analysis

• Warehouses handle different volumes of shipments.

• Average delivery time varies across warehouses but remains within a narrow range.

• Shipment costs differ across facilities, indicating operational or route differences.

• SQL joins enable the integration of warehouse operational metrics with shipment performance.

• Summary tables simplify repeated analysis and improve data accessibility.